In [ ]:
# Install dependencies
!pip install pandas networkx scikit-learn sentence-transformers


In [ ]:
import tarfile
import os

# Extract the tar.gz file
tar_path = "/content/facebook (2) (1).tar.gz"
extract_path = "/content/facebook_data"

with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(path=extract_path)


In [ ]:
import os

base_path = "/content/facebook_data"

for root, dirs, files in os.walk(base_path):
    print(f"\n📁 In directory: {root}")
    for name in files:
        print(f"📄 {name}")


In [ ]:
import os
from collections import defaultdict

# Update path to match your structure
base_path = "/content/facebook"

# Create dictionary mapping ego_id → {filetype: path}
ego_files = defaultdict(dict)

for fname in os.listdir(base_path):
    parts = fname.split('.')
    if len(parts) == 2 and parts[0].isdigit():
        ego_id, ftype = parts
        ego_files[ego_id][ftype] = os.path.join(base_path, fname)

# Show sample
print("Total ego networks found:", len(ego_files))
sample_ego_id = next(iter(ego_files))
print("Sample Ego ID:", sample_ego_id)
print("Files for that Ego:")
for ftype, path in ego_files[sample_ego_id].items():
    print(f"  {ftype}: {path}")



In [ ]:
# Load raw data
import pandas as pd

ego_id = sample_ego_id
paths = ego_files[ego_id]

# Load feature names
with open(paths['featnames'], 'r') as f:
    featnames = [line.strip().split(' ', 1)[1] for line in f]

# Load data
ego_vector = pd.read_csv(paths['egofeat'], header=None, sep=' ')
alter_vectors = pd.read_csv(paths['feat'], header=None, sep=' ')

print(f"✅ ego_vector shape: {ego_vector.shape}")
print(f"✅ alter_vectors shape: {alter_vectors.shape}")
print(f"✅ featnames count: {len(featnames)}")



In [ ]:

# Ensure alter_vectors has same number of columns as featnames
if alter_vectors.shape[1] > len(featnames):
    print("⚠️ Trimming extra column from alter_vectors")
    alter_vectors = alter_vectors.iloc[:, :len(featnames)]

# Assign feature names as column names
alter_vectors.columns = featnames
ego_vector.columns = featnames  # should already match

# Combine ego + alter
all_vectors = pd.concat([ego_vector, alter_vectors], axis=0)
print(f"✅ Combined shape: {all_vectors.shape}")


Generate SBERT Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert feature names into embeddings
feature_embeddings = model.encode(featnames)

# Function to get embedding for a node (ego or alter)
# Update get_node_embedding to safely handle extra dummy features
def get_node_embedding(binary_vector):
    active_indices = [i for i, v in enumerate(binary_vector) if v == 1 and i < len(feature_embeddings)]
    if not active_indices:
        return np.zeros_like(feature_embeddings[0])
    return np.mean([feature_embeddings[i] for i in active_indices], axis=0)

# Get ego embedding
ego_embedding = get_node_embedding(ego_vector.iloc[0].tolist())

# Get alter embeddings
alter_embeddings = [get_node_embedding(row.tolist()) for _, row in alter_vectors.iterrows()]


In [ ]:
# If alter_vectors has 1 extra column, trim it
if alter_vectors.shape[1] > len(featnames):
    print("⚠️ Trimming extra column from alter_vectors to match featnames.")
    alter_vectors = alter_vectors.iloc[:, :len(featnames)]

# Set column names properly
alter_vectors.columns = featnames
ego_vector.columns = featnames



 Compute Similarity Between Ego and Alters

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Reshape for compatibility
ego_embedding_2d = ego_embedding.reshape(1, -1)
alter_embeddings_matrix = np.vstack(alter_embeddings)

# Cosine similarities: ego vs each alter
similarities = cosine_similarity(ego_embedding_2d, alter_embeddings_matrix)[0]
avg_ego_alter_similarity = np.mean(similarities)
# Example: print average similarity
print("🔹 Average Ego-Alter Similarity:", round(np.mean(similarities), 3))


In [ ]:
import matplotlib.pyplot as plt

plt.hist(similarities, bins=30, color='skyblue')
plt.title("Ego vs Alters Cosine Similarity")
plt.xlabel("Cosine Similarity")
plt.ylabel("Frequency")
plt.show()


In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Get paths
edges_path = ego_files[ego_id]['edges']
feat_path = ego_files[ego_id]['feat']
ego_node = int(ego_id)

# Load alter–alter edge list
edges_df = pd.read_csv(edges_path, sep=' ', header=None, names=['source', 'target'])

# Get alters from the .feat file
with open(feat_path, 'r') as f:
    alter_ids = [int(line.strip().split()[0]) for line in f]

# Add ego–alter edges
ego_edges = pd.DataFrame({'source': [ego_node] * len(alter_ids), 'target': alter_ids})

# Combine with alter–alter edges
full_edges_df = pd.concat([edges_df, ego_edges], ignore_index=True)

# Build graph
G = nx.from_pandas_edgelist(full_edges_df)

# Compute layout
pos = nx.spring_layout(G, seed=42)

# Plot
plt.figure(figsize=(8, 6))
nx.draw(G, pos, node_size=20, alpha=0.6, with_labels=False)
nx.draw_networkx_nodes(G, pos, nodelist=[ego_node], node_color='red', node_size=120, label='Ego')
plt.title(f"Ego Network Structure for Ego {ego_node}")
plt.legend()
plt.show()



In [ ]:
!pip install python-igraph leidenalg


 Convert NetworkX to iGraph (Leiden requires it)

In [ ]:
import igraph as ig
import leidenalg

# Convert NetworkX graph G to iGraph
edges = list(G.edges())
G_ig = ig.Graph.TupleList(edges, directed=False)

# Optional: name nodes with IDs to keep track
G_ig.vs["name"] = list(G.nodes())


In [ ]:
# Run Leiden algorithm
partition = leidenalg.find_partition(G_ig, leidenalg.ModularityVertexPartition)

# Each node gets a community label
community_labels = partition.membership
print(f"🔍 Number of communities detected: {len(set(community_labels))}")


In [ ]:
# Map node → community
node_to_comm = {int(G_ig.vs[i]["name"]): comm for i, comm in enumerate(community_labels)}


In [ ]:
# Step 1: Prepare node_embeddings dictionary
# ego_node should be your current ego's node ID
ego_node_id = int(ego_id)  # use the actual ego_id you've defined earlier

# Assign alter_node_ids correctly
alter_node_ids = []
with open(paths['feat'], 'r') as f:
    for line in f:
        node_id = int(line.strip().split()[0])
        alter_node_ids.append(node_id)

# Ensure matching length
assert len(alter_node_ids) == len(alter_embeddings), "Mismatch in alters vs alter_embeddings length."

# Build node_embeddings dictionary
node_embeddings = {ego_node_id: ego_embedding}
node_embeddings.update({node_id: emb for node_id, emb in zip(alter_node_ids, alter_embeddings)})


In [ ]:
import numpy as np

# Group embeddings by community
community_embeddings = {}
for node, comm in node_to_comm.items():
    if node in node_embeddings:
        community_embeddings.setdefault(comm, []).append(node_embeddings[node])

# Compute centroids
community_centroids = {
    comm: np.mean(vectors, axis=0)
    for comm, vectors in community_embeddings.items()
}

print(f"✅ Computed centroids for {len(community_centroids)} communities.")


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compare ego embedding with each community centroid
ego_to_comm_sim = {
    comm: cosine_similarity(
        ego_embedding.reshape(1, -1),
        centroid.reshape(1, -1)
    )[0][0]
    for comm, centroid in community_centroids.items()
}

# Get the most similar community
primary_comm = max(ego_to_comm_sim, key=ego_to_comm_sim.get)
print(f" Ego's Primary Community: {primary_comm}")


In [ ]:
# Get centroids of other (external) communities
external_centroids = [
    centroid for comm, centroid in community_centroids.items()
    if comm != primary_comm
]

# Average similarity to external centroids
if external_centroids:
    similarities = [
        cosine_similarity(ego_embedding.reshape(1, -1), c.reshape(1, -1))[0][0]
        for c in external_centroids
    ]
    escape_score = sum(similarities) / len(similarities)
    print("🚪 Escape Potential (Echo Breaker Score):", round(escape_score, 4))
else:
    escape_score = 0.0
    print("⚠️ Only one community — Escape Potential undefined.")


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Step 1: Get sorted community IDs
comm_ids = sorted(community_centroids.keys())

# Step 2: Stack centroids into a matrix
centroid_matrix = np.stack([community_centroids[comm] for comm in comm_ids])

# Step 3: Compute cosine similarity matrix
similarity_matrix = cosine_similarity(centroid_matrix)

# Step 4: Create a labeled DataFrame for readability
similarity_df = pd.DataFrame(similarity_matrix, index=comm_ids, columns=comm_ids)

# Show
print("🔍 Inter-Community Cosine Similarity Matrix:")
display(similarity_df.round(3))  # round for readability


In [ ]:
def compute_ego_metrics(ego_id, ego_files, model):
    import pandas as pd
    import numpy as np
    import networkx as nx
    import igraph as ig
    import leidenalg
    from sklearn.metrics.pairwise import cosine_similarity

    try:
        paths = ego_files[ego_id]

        # Load feature names
        with open(paths['featnames'], 'r') as f:
            featnames = [line.strip().split(' ', 1)[1] for line in f]

        # Load ego and alter features
        ego_vector = pd.read_csv(paths['egofeat'], header=None, sep=' ')
        alter_vectors = pd.read_csv(paths['feat'], header=None, sep=' ')

        if alter_vectors.shape[1] > len(featnames):
            alter_vectors = alter_vectors.iloc[:, :len(featnames)]

        ego_vector.columns = featnames
        alter_vectors.columns = featnames

        # Convert feature names to embeddings
        feature_embeddings = model.encode(featnames)

        def get_node_embedding(binary_vector):
            active_indices = [i for i, v in enumerate(binary_vector) if v == 1 and i < len(feature_embeddings)]
            if not active_indices:
                return np.zeros_like(feature_embeddings[0])
            return np.mean([feature_embeddings[i] for i in active_indices], axis=0)

        # Get embeddings
        ego_embedding = get_node_embedding(ego_vector.iloc[0].tolist())
        alter_embeddings = [get_node_embedding(row.tolist()) for _, row in alter_vectors.iterrows()]

        # Ego–Alter similarity
        similarities = cosine_similarity(ego_embedding.reshape(1, -1), np.vstack(alter_embeddings))[0]
        avg_ego_alter_similarity = np.mean(similarities)

        # Build graph
        G = nx.read_edgelist(paths['edges'], nodetype=int)
        G_ig = ig.Graph.TupleList(list(G.edges()), directed=False)
        G_ig.vs["name"] = list(G.nodes())

        # Leiden algorithm
        partition = leidenalg.find_partition(G_ig, leidenalg.ModularityVertexPartition)
        community_labels = partition.membership
        node_to_comm = {int(G_ig.vs[i]["name"]): comm for i, comm in enumerate(community_labels)}

        # Map nodes to embeddings
        ego_node_id = int(ego_id)
        with open(paths['feat'], 'r') as f:
            alter_node_ids = [int(line.strip().split()[0]) for line in f]

        node_embeddings = {ego_node_id: ego_embedding}
        node_embeddings.update({nid: emb for nid, emb in zip(alter_node_ids, alter_embeddings)})

        # Compute community centroids
        community_embeddings = {}
        for node, comm in node_to_comm.items():
            if node in node_embeddings:
                community_embeddings.setdefault(comm, []).append(node_embeddings[node])
        community_centroids = {comm: np.mean(vecs, axis=0) for comm, vecs in community_embeddings.items()}

        # Primary community
        ego_to_comm_sim = {
            comm: cosine_similarity(ego_embedding.reshape(1, -1), centroid.reshape(1, -1))[0][0]
            for comm, centroid in community_centroids.items()
        }
        primary_comm = max(ego_to_comm_sim, key=ego_to_comm_sim.get)

        # Escape potential
        external_centroids = [centroid for comm, centroid in community_centroids.items() if comm != primary_comm]
        if external_centroids:
            escape_score = np.mean([
                cosine_similarity(ego_embedding.reshape(1, -1), c.reshape(1, -1))[0][0]
                for c in external_centroids
            ])
        else:
            escape_score = 0.0

        # Inter-community similarity
        comm_ids = sorted(community_centroids.keys())
        centroid_matrix = np.stack([community_centroids[comm] for comm in comm_ids])
        inter_comm_sim = cosine_similarity(centroid_matrix)
        avg_inter_comm_similarity = np.mean(inter_comm_sim[np.triu_indices_from(inter_comm_sim, k=1)])

        return {
            "ego_id": ego_id,
            "avg_ego_alter_similarity": avg_ego_alter_similarity,
            "num_communities": len(community_centroids),
            "primary_community_size": len(community_embeddings[primary_comm]),
            "escape_potential": escape_score,
            "avg_inter_community_similarity": avg_inter_comm_similarity
        }

    except Exception as e:
        print(f"❌ Error processing ego {ego_id}: {e}")
        return None




In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd

# Load SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Collect metrics for all egos
all_ego_metrics = []

for ego_id in ego_files.keys():
    result = compute_ego_metrics(ego_id, ego_files, model)
    if result:
        all_ego_metrics.append(result)

# Convert to DataFrame
metrics_df = pd.DataFrame(all_ego_metrics)
metrics_df.head()


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X= scaler.fit_transform(metrics_df.drop(columns=["ego_id"]))


In [ ]:
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=3, random_state=42)
metrics_df["gmm_cluster"] = gmm.fit_predict(X)

from sklearn.metrics import silhouette_score
sil_score = silhouette_score(X, metrics_df["gmm_cluster"])
print("GMM Silhouette Score:", round(sil_score, 3))

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

pca = PCA(n_components=2)
pca_result = pca.fit_transform(X)
metrics_df["pca1"] = pca_result[:, 0]
metrics_df["pca2"] = pca_result[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=metrics_df, x="pca1", y="pca2", hue="gmm_cluster", palette="Set2", s=100)
plt.title("GMM Behavioral Clusters of Egos")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.legend(title="Cluster")
plt.grid(True)
plt.show()

In [ ]:


from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 2. KMeans clustering
kmeans = KMeans(n_clusters=3, random_state=42)
metrics_df['cluster'] = kmeans.fit_predict(X)

# 3. PCA for 2D visualization
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X)
metrics_df['pca1'] = pca_result[:, 0]
metrics_df['pca2'] = pca_result[:, 1]

# 4. Plot the clusters
plt.figure(figsize=(8, 6))
sns.scatterplot(data=metrics_df, x='pca1', y='pca2', hue='cluster', palette='tab10', s=100)
plt.title("Behavioral Clustering of Egos (KMeans)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.legend(title='Cluster')
plt.grid(True)
plt.show()

In [ ]:
import hdbscan
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt

# Fit HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size=5)
metrics_df['hdbscan_cluster'] = clusterer.fit_predict(X)

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X)
metrics_df['pca1'] = pca_result[:, 0]
metrics_df['pca2'] = pca_result[:, 1]

# Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(data=metrics_df, x='pca1', y='pca2', hue='hdbscan_cluster', palette='Set1', s=100)
plt.title("HDBSCAN Behavioral Clustering")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.legend(title='Cluster')
plt.grid(True)
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# Run DBSCAN
dbscan = DBSCAN(eps=0.8, min_samples=3)  # You can tune eps and min_samples
metrics_df['dbscan_cluster'] = dbscan.fit_predict(X)
# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X)
metrics_df['pca1'] = pca_result[:, 0]
metrics_df['pca2'] = pca_result[:, 1]

# Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=metrics_df, x='pca1', y='pca2',
    hue='dbscan_cluster', palette='tab10', s=100
)
plt.title("Behavioral Clustering of Egos (DBSCAN)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.legend(title='Cluster')
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score

# KMeans clustering
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X)

# Calculate Silhouette Score
score = silhouette_score(X, labels)
print(f"KMeans Silhouette Score: {score:.3f}")
gmm_labels = metrics_df['gmm_cluster']
score_gmm = silhouette_score(X, gmm_labels)
print(f"GMM Silhouette Score: {score_gmm:.3f}")
dbscan_labels = metrics_df['dbscan_cluster']
# Check if DBSCAN returned more than 1 cluster (excluding -1/noise)
if len(set(dbscan_labels)) > 1:
    score_dbscan = silhouette_score(X, dbscan_labels)
    print(f"DBSCAN Silhouette Score: {score_dbscan:.3f}")
else:
    print("DBSCAN Silhouette Score: Not valid (only one cluster or mostly noise)")
    hdbscan_labels = metrics_df['hdbscan_cluster']
if len(set(hdbscan_labels)) > 1:
    score_hdbscan = silhouette_score(X, hdbscan_labels)
    print(f"HDBSCAN Silhouette Score: {score_hdbscan:.3f}")
else:
    print("HDBSCAN Silhouette Score: Not valid (only one cluster or mostly noise)")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If you already created this mapping:
# metrics_df['label'] = metrics_df['cluster'].map({0: "Echo Chamber", 1: "Echo Breaker", 2: "Bridge"})

# Create the 'label' column by mapping GMM cluster numbers to descriptive names
# You might need to adjust the mapping based on your analysis of the clusters
cluster_mapping = {
    0: "Cluster 0",
    1: "Cluster 1",
    2: "Cluster 2"
}
metrics_df['label'] = metrics_df['gmm_cluster'].map(cluster_mapping)


plt.figure(figsize=(10, 7))
sns.scatterplot(data=metrics_df, x='pca1', y='pca2', hue='label', palette='Set2', s=100)

plt.title("Clustering of Egos Based on Behavioral Metrics", fontsize=14)
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(title='Behavioral Cluster')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Define layout
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.suptitle("Clustering of Egos Based on Behavioral Metrics (All Methods)", fontsize=16)

# KMeans Plot
sns.scatterplot(
    ax=axes[0, 0],
    data=metrics_df,
    x='pca1', y='pca2',
    hue='label',
    palette='tab10',
    s=100
)
axes[0, 0].set_title("KMeans Clustering")
axes[0, 0].set_xlabel("PCA 1")
axes[0, 0].set_ylabel("PCA 2")
axes[0, 0].legend(title="label")

# GMM Plot
sns.scatterplot(
    ax=axes[0, 1],
    data=metrics_df,
    x='pca1', y='pca2',
    hue='gmm_cluster',
    palette='Set2',
    s=100
)
axes[0, 1].set_title("GMM Clustering")
axes[0, 1].set_xlabel("PCA 1")
axes[0, 1].set_ylabel("PCA 2")
axes[0, 1].legend(title="Cluster")

# DBSCAN Plot
sns.scatterplot(
    ax=axes[1, 0],
    data=metrics_df,
    x='pca1', y='pca2',
    hue='dbscan_cluster',
    palette='tab20',
    s=100
)
axes[1, 0].set_title("DBSCAN Clustering")
axes[1, 0].set_xlabel("PCA 1")
axes[1, 0].set_ylabel("PCA 2")
axes[1, 0].legend(title="label")

# HDBSCAN Plot
sns.scatterplot(
    ax=axes[1, 1],
    data=metrics_df,
    x='pca1', y='pca2',
    hue='hdbscan_cluster',
    palette='Set1',
    s=100
)
axes[1, 1].set_title("HDBSCAN Clustering")
axes[1, 1].set_xlabel("PCA 1")
axes[1, 1].set_ylabel("PCA 2")
axes[1, 1].legend(title="label")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
metrics_df.groupby("label").mean(numeric_only=True).round(3)
